<a href="https://colab.research.google.com/github/Santiago-Soria/proyecto-transformacion-texto-imagen/blob/sant_branch/notebooks/3.2_run_experimentos_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Clonar repositorio
!git clone https://github.com/Santiago-Soria/proyecto-transformacion-texto-imagen.git


Cloning into 'proyecto-transformacion-texto-imagen'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 158 (delta 61), reused 117 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (158/158), 1.67 MiB | 24.11 MiB/s, done.
Resolving deltas: 100% (61/61), done.


In [3]:
# Cambiar al directorio del proyecto
%cd proyecto-transformacion-texto-imagen

/content/proyecto-transformacion-texto-imagen


In [4]:
# Instalar dependencias del proyecto
!pip install transformers datasets torch scikit-learn pandas numpy spacy importlib matplotlib-venn
!python -m spacy download es_core_news_sm

  Preparing metadata (setup.py) ... done
  Created wheel for importlib: filename=importlib-1.0.4-py3-none-any.whl size=5850 sha256=26104af2f135d0c5e93e6c650b85a7bb3ec9de8ebc910a82022c4ea2c75665c6
  Stored in directory: /root/.cache/pip/wheels/40/41/c4/d925a53b7b7e75a65369e1b17f7bade00d7907ac5a7d74dc5f
Successfully built importlib
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 116.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import torch
print(f"GPU disponible: {torch.cuda.is_available()}")
print(f"Dispositivo: {torch.cuda.get_device_name(0)}")

GPU disponible: True
Dispositivo: Tesla T4


In [ ]:
import sys
import os
import importlib
# Agregar src al path desde la raíz del proyecto
sys.path.insert(0, '/content/proyecto-transformacion-texto-imagen/src')

from models import train_classic
# Recargar el módulo
importlib.reload(train_classic)

# Ahora importar normalmente
import pandas as pd
from preprocessing_utils import preprocess_text
from features.extraction import FeatureExtractor
from models.train_classic import train_logistic
from models.train_transformer import run_finetuning

In [ ]:
from re import X
train = pd.read_csv('/content/proyecto-transformacion-texto-imagen/data/processed/train.csv')
test = pd.read_csv('/content/proyecto-transformacion-texto-imagen/data/processed/test.csv')

# Limpieza Mínima en el Preprocesamiento
X_train_p1 = [preprocess_text(t, 'P1') for t in train['text']]
X_test_p1 = [preprocess_text(t, 'P1') for t in test['text']]

# Extractor de Caractetísticas
extractor =  FeatureExtractor()

### **Experimento 2.2:** Limpieza Básica + RoBERTa-BNE (Frozen) + Regresión Logística

In [ ]:
# ==========================================
# EXP 2.2: Frozen RoBERTa + LR
# ==========================================
# Usamos P1 (Transformers prefieren texto con estructura, no P2)
print("Generando embeddings. . .")
X_train_emb = extractor.get_transformer_embeddings(X_train_p1, "bertin-project/bertin-roberta-base-spanish")
X_test_emb = extractor.get_transformer_embeddings(X_test_p1, "bertin-project/bertin-roberta-base-spanish")

models_dir = '/content/proyecto-transformacion-texto-imagen/models'
os.makedirs(models_dir, exist_ok=True)

train_logistic(X_train_emb, train['manual_classification'],
               X_test_emb, test['manual_classification'], "Exp_2.2_RoBERTa_Frozen")

Generando embeddings. . .
--> Extrayendo embeddings con bertin-project/bertin-roberta-base-spanish...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: bertin-project/bertin-roberta-base-spanish
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 29/29 [00:06<00:00,  4.20it/s]


--> Extrayendo embeddings con bertin-project/bertin-roberta-base-spanish...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: bertin-project/bertin-roberta-base-spanish
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 4/4 [00:00<00:00,  4.37it/s]


Entrenando Regresión Logística para Exp_2.2_RoBERTa_Frozen...
--- Resultados Exp_2.2_RoBERTa_Frozen ---
              precision    recall  f1-score   support

           0       0.71      0.66      0.68        70
           1       0.51      0.57      0.54        44

    accuracy                           0.62       114
   macro avg       0.61      0.61      0.61       114
weighted avg       0.63      0.62      0.63       114



(LogisticRegression(class_weight='balanced', random_state=42, solver='liblinear'),
 array([0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0,
        1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1,
        1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0,
        0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0,
        1, 0, 1, 1]))

### **Experimento 2.3:** Limpieza Básica + mDeBERTa-v3-base (Frozen) + Regresión Logística

In [ ]:
# ==========================================
# EXP 2.3: Frozen mDeBERTa-v3-base + LR
# ==========================================
# Usamos P1 (Transformers prefieren texto con estructura, no P2)
print("Generando embeddings. . .")
X_train_emb = extractor.get_transformer_embeddings(X_train_p1, "microsoft/mdeberta-v3-base")
X_test_emb = extractor.get_transformer_embeddings(X_test_p1, "microsoft/mdeberta-v3-base")

models_dir = '/content/proyecto-transformacion-texto-imagen/models'
os.makedirs(models_dir, exist_ok=True)

train_logistic(X_train_emb, train['manual_classification'],
               X_test_emb, test['manual_classification'], "Exp_2.3_mDeBERTa_Frozen")

Generando embeddings. . .
--> Extrayendo embeddings con microsoft/mdeberta-v3-base...


pytorch_model.bin:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.dense.weight              | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/ar

--> Extrayendo embeddings con microsoft/mdeberta-v3-base...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.dense.weight              | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/ar

Entrenando Regresión Logística para Exp_2.3_mDeBERTa_Frozen...
--- Resultados Exp_2.3_mDeBERTa_Frozen ---
              precision    recall  f1-score   support

           0       0.63      0.64      0.64        70
           1       0.42      0.41      0.41        44

    accuracy                           0.55       114
   macro avg       0.53      0.53      0.53       114
weighted avg       0.55      0.55      0.55       114



(LogisticRegression(class_weight='balanced', random_state=42, solver='liblinear'),
 array([1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1,
        0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0,
        0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0,
        1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1,
        1, 0, 0, 1]))

### **Experimento 3.2:** Limpieza Básica + RoBERTa-BNE(Fine-Tuning) + Softmax

In [ ]:
# ==========================================
# EXP 3.2: Fine-Tuning RoBERTa
# ==========================================
# Este es el pesado. Asegúrate de estar en GPU.
trainer = run_finetuning(X_train_p1, train['manual_classification'].values,
                         X_test_p1, test['manual_classification'].values,
                         "bertin-project/bertin-roberta-base-spanish")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: bertin-project/bertin-roberta-base-spanish
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'